# Parse market
Fold sorted parsed FIX messages into books, or write their orders and executions directly.

In [ ]:
project_root = "."
fix_dictionary = None
source = "fix.market"
books = True
target = "market.books"
order_target = "market.orders"
execution_target = "market.executions"
start = None
end = None
catalog = {"name": "rekep", "properties": {}}
table_properties = {"history.expire.max-snapshot-age-ms": "604800000"}
branch = "root"
snapshot_every = 3_600_000_000_000
max_lateness_ns = 900_000_000_000
max_order_age_ns = 86_400_000_000_000
max_side_alive = 10_000
merge_by = True
batch_row_size = 65_536
commit_batch_num = 8
commit_row_size = None
log_level = "INFO"

In [ ]:
import pyarrow.compute as pc
from pyiceberg.expressions import (
    And,
    GreaterThanOrEqual,
    IsNull,
    LessThan,
    LessThanOrEqual,
    NotEqualTo,
    NotNull,
)

from rekep.enums import EventType
from rekep.fix.registry import FixRegistry
from rekep.iceberg import IcebergCatalog
from rekep.logs import Stage, configure
from rekep.market import Book, BookIterator, Execution, Order
from rekep.text import FixMsg
from rekep.times import unix_of
from rekep.urls import Url

configure(log_level)
if isinstance(batch_row_size, bool) or not isinstance(batch_row_size, int):
    raise TypeError("batch_row_size must be an integer")
if batch_row_size <= 0:
    raise ValueError("batch_row_size must be positive")
if isinstance(commit_batch_num, bool) or not isinstance(commit_batch_num, int):
    raise TypeError("commit_batch_num must be an integer")
if commit_batch_num <= 0:
    raise ValueError("commit_batch_num must be positive")
if commit_row_size is not None and (
    isinstance(commit_row_size, bool) or not isinstance(commit_row_size, int)
):
    raise TypeError("commit_row_size must be an integer or null")
if commit_row_size is not None and commit_row_size <= 0:
    raise ValueError("commit_row_size must be positive")
HOUR = 3_600_000_000_000
DAY = 86_400_000_000_000


configured_targets = (
    (target,)
    if books
    else tuple(name for name in (order_target, execution_target) if name is not None)
)
if not configured_targets or configured_targets[0] is None:
    raise ValueError("book mode needs target; direct mode needs an event target")
if len(configured_targets) != len(set(configured_targets)):
    raise ValueError("market targets must be distinct")


def _window(lower, upper, column="unix"):
    predicates = []
    if lower is not None:
        predicates.append(GreaterThanOrEqual(column, lower))
    if upper is not None:
        predicates.append(LessThan(column, upper))
    return None if not predicates else predicates[0] if len(predicates) == 1 else And(*predicates)


lower, upper = unix_of(start), unix_of(end, upper=True)
read_lower = None if lower is None else lower - lower % HOUR - HOUR
read_upper = None if upper is None else ((upper + HOUR - 1) // HOUR) * HOUR + max_lateness_ns
# The targets a run actually writes, keyed by what each holds. A role this
# mode does not write is left out rather than stored as null.
stage = Stage(
    "parse_market",
    sources={"market": source},
    targets=(
        {"books": target}
        if books
        else {
            role: name
            for role, name in (("orders", order_target), ("executions", execution_target))
            if name is not None
        }
    ),
    window=(lower, upper),
)
registry = (
    FixRegistry()
    if fix_dictionary is None
    else FixRegistry(cache_dir=Url.from_string(str(fix_dictionary)).resolve(project_root))
)
store = IcebergCatalog.from_dict(catalog)
logs_table = store.dataset(
    source,
    field=FixMsg.into_field(),
    branch=branch,
)
book_table = (
    None
    if not books
    else store.dataset(
        target,
        field=Book.into_field(),
        table_properties=dict(table_properties),
        branch=branch,
        commit_batch_num=commit_batch_num,
        commit_row_size=commit_row_size,
    )
)


def _event_table(name, event_type):
    if name is None:
        return None
    return store.dataset(
        name,
        field=event_type.into_field(),
        table_properties=dict(table_properties),
        branch=branch,
        commit_batch_num=commit_batch_num,
        commit_row_size=commit_row_size,
    )


order_table = None if books else _event_table(order_target, Order)
execution_table = None if books else _event_table(execution_target, Execution)


def _book_seeds():
    if book_table is None or read_lower is None:
        return ()
    recent = And(
        GreaterThanOrEqual("unix", read_lower - DAY),
        LessThanOrEqual("unix", read_lower),
        NotNull("snapunix"),
    )
    reader = book_table.read_arrow_reader(
        Book.into_field(), row_filter=recent, order_by=("unix", "hash")
    )
    return Book.from_arrow_reader(reader)


def _log_reader():
    row_filter = _window(read_lower, read_upper)
    if book_table is None:
        market_events = NotEqualTo("eventtype", int(EventType.INSTRUMENT))
        row_filter = market_events if row_filter is None else And(row_filter, market_events)
    # parse_fix_market retains failed rows in its source category for audit. They
    # cannot mutate a book or emit a partial order from an incomplete reading.
    clean = IsNull("error")
    row_filter = clean if row_filter is None else And(row_filter, clean)
    reader = logs_table.read_arrow_reader(
        FixMsg.into_field(),
        row_filter=row_filter,
        order_by=("unix", "msgseqnum", "hash"),
    )
    return reader


def _inside(event):
    return (lower is None or event.unix >= lower) and (upper is None or event.unix < upper)


def _write_books():
    snapshots = _book_seeds()
    read = {"books": 0, "orders": 0, "executions": 0}
    events = FixMsg.into_ordered_market_events(_log_reader(), registry=registry)
    iterating = BookIterator.from_events(
        events,
        snapshots=snapshots,
        registry=registry,
        snapshot_every=snapshot_every,
        snapshot_until=upper,
        max_order_age_ns=max_order_age_ns,
        max_side_alive=max_side_alive,
    )

    def selected():
        nonlocal read
        for book in iterating:
            if _inside(book):
                read["books"] += 1
                read["orders"] += len(book.deltas)
                read["executions"] += len(book.executions)
                yield book

    written = book_table.append_arrow_reader(
        Book.into_arrow_reader(selected(), batch_row_size=batch_row_size),
        Book.into_field(),
        merge_by=merge_by,
        commit_row_size=commit_row_size,
        commit_batch_num=commit_batch_num,
    )
    checkpoint = min((book.unix for book in iterating.snapshots), default=None)
    return read, written, checkpoint


def _write_events():
    tables = {Order: order_table, Execution: execution_table}
    read = {Order: 0, Execution: 0}
    written = {Order: 0, Execution: 0}
    buffers = {event_type: [] for event_type in tables}
    held_rows = {event_type: 0 for event_type in tables}

    def flush(event_type):
        batches = buffers[event_type]
        table = tables[event_type]
        if not batches or table is None:
            return
        written[event_type] += table.append_arrow_reader(
            iter(batches),
            event_type.into_field(),
            merge_by=merge_by,
            commit_row_size=commit_row_size,
            commit_batch_num=commit_batch_num,
        )
        buffers[event_type] = []
        held_rows[event_type] = 0

    for event_type, batch in FixMsg.into_market_arrow_batches(
        _log_reader(), batch_row_size=batch_row_size, registry=registry
    ):
        mask = None
        if lower is not None:
            mask = pc.greater_equal(batch.column("unix"), lower)
        if upper is not None:
            before = pc.less(batch.column("unix"), upper)
            mask = before if mask is None else pc.and_(mask, before)
        if mask is not None:
            batch = batch.filter(mask)
        read[event_type] += batch.num_rows
        table = tables[event_type]
        if table is None or not batch.num_rows:
            continue
        buffers[event_type].append(batch)
        held_rows[event_type] += batch.num_rows
        row_cap = commit_row_size is not None and held_rows[event_type] >= commit_row_size
        if len(buffers[event_type]) >= commit_batch_num or row_cap:
            flush(event_type)
    for event_type in tables:
        flush(event_type)
    return read, written


read = {"books": 0, "orders": 0, "executions": 0}
written = dict(read)
flatten = {"orders": 0, "executions": 0}
checkpoint = None
if book_table is not None:
    book_read, written["books"], checkpoint = _write_books()
    read.update(book_read)
    flatten.update({name: book_read[name] for name in flatten})
else:
    event_read, event_written = _write_events()
    read["orders"], read["executions"] = event_read[Order], event_read[Execution]
    written["orders"], written["executions"] = (
        event_written[Order],
        event_written[Execution],
    )
mode = "books" if book_table is not None else "events"
stage.says(
    "folded in %s mode: %s",
    mode,
    ", ".join(f"{count} {product}" for product, count in read.items() if count),
)
# One number each, like every other stage: what this stage produced. In book
# mode that is books, and the orders and executions nested inside them are
# what the two flatteners read; in direct mode it is the events themselves.
# `products` keeps the breakdown either way.
result = stage.finished(
    read=read["books"] if books else read["orders"] + read["executions"],
    written=written["books"] if books else written["orders"] + written["executions"],
    mode=mode,
    products={"read": read, "written": written},
    flatten=flatten,
    checkpoint=checkpoint,
    scan={"start": read_lower, "end": read_upper},
)
result